# 现代前端第一步——模块化

### 模块化解决相同变量名和顺序的问题

## 先运行一个变大的项目

课程提供了配套示例代码，可以先拉到本地体验(可以在本地电脑的家目录下拉取然后将新内容复制到原项目中)：

```bash
git clone https://github.com/joylibo/zero-to-tech-demos.git
cd zero-to-tech-demos/zero-to-tech-4-1
```

打开 `zero-to-tech-4-1/` 后，可以看到下面的文件：

```text
zero-to-tech-4-1/
├── index.html
├── text-lab.html
├── css/
│   └── ... 约 8 个 CSS 文件
└── js/
    ├── cards.js   卡片飞入动画
    ├── score.js   分数动画
    └── nav.js     当前页面导航高亮
```

页面增加后，HTML 里的资源引用也变长：

```html
<link rel="stylesheet" href="css/nav.css">
<link rel="stylesheet" href="css/cards.css">
<!-- ... 更多 CSS -->
<script src="js/cards.js"></script>
<script src="js/score.js"></script>
<script src="js/nav.js"></script>
```

## 本地脚本与第三方 CDN 资源

项目中的脚本通常来自两处：

```html
<!-- 本地文件 -->
<script src="js/cards.js"></script>

<!-- 互联网上的第三方文件 -->
<script src="https://cdn.jsdelivr.net/npm/animejs@4/lib/anime.iife.min.js"></script>
```

浏览器看来，两者都是“加载一个 JavaScript 文件”：前者从项目目录读取，后者通过 URL 从 CDN 读取。CDN 是内容分发网络，别人已经把库托管在公开服务器上，我们直接引用即可。

Anime.js 是一个动画库，提供 `anime.animate`、`anime.stagger` 等动画能力。库本身只提供工具，具体应用仍由项目自己的 `cards.js` 调用。

## 模块化的两个关键词： import 与 export

模块化把全局“全局”改成“局部”：

| 关键词 | 作用 |
|------|------|
| `export` | 明确标记哪些函数或数据允许其他文件使用 |
| `import` | 明确声明当前文件要使用哪个文件导出的内容 |

依赖关系写在代码中，浏览器就能根据 `import` 自动计算加载顺序；每个模块的变量默认是局部的，也不会因为同名而互相覆盖。

## ES、ES6 与 ES Modules

- **ES** 是 ECMAScript 的缩写，是 JavaScript 的官方标准规范。
- **ES6** 是 ECMAScript 第 6 版，2015 年发布。
- `import` 和 `export` 在 ES6 中被正式定义。
- **ES Modules** 就是 JavaScript 标准内置的模块系统，不是另一种语言。

> 看到“ES”，可以先理解成“JavaScript 官方标准”,用了 ES 模块就不能本地打开了。

## 把三个脚本改成模块

### 1. cards.js：自己声明依赖

```javascript
import { animate, stagger } from
  "https://cdn.jsdelivr.net/npm/animejs@4/+esm";

export function initCardsAnim() {
  animate(".card", {
    opacity: [0, 1],
    translateY: [24, 0],
    delay: stagger(120),
    duration: 700,
    ease: "outBack"
  });
}
```

`cards.js` 不再依赖全局的 `anime`，而是在文件顶部明确导入 `animate` 和 `stagger`；`initCardsAnim` 用 `export` 标记，供入口文件调用。`+esm` 表示 Anime.js 的 ES 模块版本。

### 2. score.js：自己声明依赖

```javascript
import { animate, scrambleText } from
  "https://cdn.jsdelivr.net/npm/animejs@4/+esm";

export function initScoreAnim() {
  const btn = document.querySelector(".primary-button");
  const scoreEl = document.querySelector("[data-score]");
  if (!btn || !scoreEl) return;

  btn.addEventListener("click", () => {
    animate(scoreEl, {
      innerHTML: scrambleText({ chars: "0-9" }),
      duration: 1500
    });
  });
}
```

数字动画同样从 ES 模块版本导入自己的依赖。没有 `.primary-button` 或 `[data-score]` 的页面会直接返回，不影响个人主页。

### 3. nav.js：自己声明依赖

改造后，把它变成可被入口文件调用的模块函数：

```javascript
export function initNav() {
  const path = location.pathname.split("/").pop() || "index.html";
  const links = document.querySelectorAll(".nav-link");

  for (const link of links) {
    const href = link.getAttribute("href");
    link.classList.toggle("active", href === path);
  }
}
```

### 4. 用 main.js 统一管理入口

在 `js/` 目录中新建 `main.js`：

```javascript
import { initNav } from "./nav.js";
import { initCardsAnim } from "./cards.js";
import { initScoreAnim } from "./score.js";

initNav();
initCardsAnim();
initScoreAnim();
```

`main.js` 是应用入口：它列出三个模块的依赖，再统一调用初始化函数。依赖关系和执行关系从 HTML 的隐藏顺序，变成了代码中的明文声明。

### 5. 运用新的模块化方法

然后在 `index.html` 和 `text-lab.html` 中，删除原来的多行 `<script>`，只保留：

```html
<script type="module" src="js/main.js"></script>
```

`type="module"` 告诉浏览器：这是一个 ES 模块入口，需要按 `import` 继续加载它依赖的文件。